# Retail Lens: Sales performance Extension

This notebook extends the original analysis. Its focus is the sales performance, which categories contribute revenue, how monthly patterns differ, and how changes in transaction count and average purchase combine into changes in revenue. The included website makes these questions explorable through the different filters and clickable charts.

Website:

In [3]:
from pathlib import Path
import calendar
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

## 1. Establish a trustworthy baseline:
We can check the dates, identifiers, numeric inputs and transaction arithmetic before drawing charts. An average price per row is not the same as revenue per unit: the latter must be weighted by quantity.

In [6]:
required = ["Transaction ID", "Date", "Customer ID", "Gender", "Age", "Product Category", "Quantity", "Price per Unit", "Total Amount"]
assert set(required).issubset(df.columns), "Missing required columns"
assert not df[required].isna().any().any(), "Missing data"
assert df["Transaction ID"].is_unique, "Repeated transaction IDs"
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors="raise")
assert (df["Quantity"] > 0).all() and (df["Quantity"] % 1 == 0).all()
assert (df["Price per Unit"] > 0).all()
assert ((df["Quantity"] * df["Price per Unit"] - df["Total Amount"]).abs() < .01).all()
summary = pd.Series({
    "Revenue": df["Total Amount"].sum(),
    "Transactions": len(df),
    "Units": df["Quantity"].sum(),
    "Average purchase": df["Total Amount"].mean(),
    "Revenue per unit": df["Total Amount"].sum() / df["Quantity"].sum()})
display(summary.to_frame("Value").round(2))
print(f"Date coverage: {df.Date.min().date()} to {df.Date.max().date()}")

,Value
Revenue,456000.00
Transactions,1000.00
Units,2514.00
Average purchase,456.00
Revenue per unit,181.38


Date coverage: 2023-01-01 to 2024-01-01


## 2. Query with SQL and reconcile with Python:
SQLite is included in Python. Using both SQL and Pandas gives a useful independent check on the headline totals. For bundled data, expect 456,000 in revenue, 1,000 transactions and 2,514 units.

In [7]:
connection = sqlite3.connect(":memory:")
sql_df = df.copy()
sql_df["Date"] = sql_df.Date.dt.strftime("%Y-%m-%d")
sql_df.to_sql("sales", connection, index=False, if_exists="replace")
category_query = '-- Weighted unit value must use SUM(revenue) / SUM(quantity).\nSELECT\n    "Product Category" AS category,\n    SUM("Total Amount") AS revenue,\n    SUM("Quantity") AS units,\n    COUNT(*) AS transactions,\n    ROUND(1.0 * SUM("Total Amount") / SUM("Quantity"), 2) AS revenue_per_unit,\n    ROUND(AVG("Total Amount"), 2) AS average_purchase\nFROM sales\nGROUP BY "Product Category"\nORDER BY revenue DESC;\n'
category = pd.read_sql_query(category_query, connection)
assert category.revenue.sum() == df["Total Amount"].sum()
assert category.transactions.sum() == len(df)
assert category.units.sum() == df.Quantity.sum()
category["revenue_share_pct"] = 100 * category.revenue / category.revenue.sum()
display(category.round(2))

,category,revenue,units,transactions,revenue_per_unit,average_purchase,revenue_share_pct
0,Electronics,156905,849,342,184.81,458.79,34.41
1,Clothing,155580,894,351,174.03,443.25,34.12
2,Beauty,143515,771,307,186.14,467.48,31.47


## 3. Category scale and value:
The bar chart shows total revenue. The scatter separates units sold from revenue per unit; bubble area represents revenue. These are accounting relationships, not evidence that one variable causes another.

In [11]:
fig = px.bar(category.sort_values("revenue"), x="revenue", y="category",
             orientation="h", color="category", text_auto=",.0f",
             title="Category contribution - all available dates:",
             labels={"revenue": "Revenue (currency unspecified)", "category": ""})
fig.update_layout(showlegend=False, height=350)
fig.show()
fig = px.scatter(category, x="units", y="revenue_per_unit", size="revenue",
                 color="category", text="category", size_max=65,
                 hover_data=["transactions", "average_purchase", "revenue_share_pct"],
                 labels={"units": "Units sold", "revenue_per_unit": "Revenue per unit"},
                 title="Sales volume and weighted unit value:")
fig.update_traces(textposition="top center")
fig.update_layout(height=420, showlegend=False)
fig.show()

## 4. Compare complete months fairly:
We can use the January - December 2023 for monthly comparisons. The file includes only the January 1 of 2024, so displaying January 2024 as a complete month would be misleading. Missing transaction days within 2023 count as zero sales, daily averages divide by all calendar days, not just days with transactions. This assumes the file covers the year completely.

In [13]:
monthly_query = '-- January 2024 is excluded because the original file ends on January 1.\nSELECT\n    strftime(\'%Y-%m\', Date) AS month,\n    SUM("Total Amount") AS revenue,\n    COUNT(*) AS transactions,\n    SUM(Quantity) AS units,\n    ROUND(AVG("Total Amount"), 2) AS average_purchase\nFROM sales\nWHERE Date >= \'2023-01-01\' AND Date < \'2024-01-01\'\nGROUP BY strftime(\'%Y-%m\', Date)\nORDER BY month;\n'
monthly = pd.read_sql_query(monthly_query, connection)
monthly["calendar_days"] = monthly.month.map(
    lambda x: calendar.monthrange(int(x[:4]), int(x[5:]))[1])
monthly["revenue_per_calendar_day"] = monthly.revenue / monthly.calendar_days
year = df[df.Date.between("2023-01-01", "2023-12-31")].copy()
assert monthly.revenue.sum() == year["Total Amount"].sum()
display(monthly.round(2))
fig = px.line(monthly, x="month", y="revenue_per_calendar_day", markers=True,
              title="Revenue per calendar day - complete months in 2023:",
              hover_data=["revenue", "transactions", "calendar_days"],
              labels={"month": "Month", "revenue_per_calendar_day": "Revenue per calendar day"})
fig.update_layout(height=400)
fig.show()

,month,revenue,transactions,units,average_purchase,calendar_days,revenue_per_calendar_day
0,2023-01,35450,76,195,466.45,31,1143.55
1,2023-02,44060,85,214,518.35,28,1573.57
2,2023-03,28990,73,194,397.12,31,935.16
3,2023-04,33870,86,214,393.84,30,1129.00
4,2023-05,53150,105,259,506.19,31,1714.52
5,2023-06,36715,77,197,476.82,30,1223.83
6,2023-07,35465,72,176,492.57,31,1144.03
7,2023-08,36960,94,227,393.19,31,1192.26
8,2023-09,23620,65,170,363.38,30,787.33
9,2023-10,46580,96,252,485.21,31,1502.58
